In [1]:
foo = "https://openveda.cloud/api/features/collections/public.eis_fire_lf_fireline_nrt/items?f=geojson&sortby=-t"

In [2]:
import requests

import sys
import requests
from datetime import datetime, timezone, timedelta
import pytz

In [3]:
foo = requests.get("https://openveda.cloud/api/features/collections/public.eis_fire_lf_fireline_nrt/items?f=geojson&sortby=-t")

In [4]:
foo

<Response [200]>

In [5]:
data = foo.json()

In [6]:
# def get_most_recent_timestep(data: dict | list, timestamp_field: str, data_field: str = None) -> dict:
#     """
#     Parse the JSON response and return the record with the most recent timestamp.

#     Args:
#         data:            The parsed JSON response (dict or list).
#         timestamp_field: The key name of the timestamp in each record.
#         data_field:      If the records are nested under a key in a dict, provide that key.

#     Returns:
#         The record with the most recent timestamp.
#     """
#     # If the data is wrapped in a parent dict, extract the list
#     if isinstance(data, dict) and data_field:
#         records = data.get(data_field, [])
#     elif isinstance(data, list):
#         records = data
#     else:
#         raise ValueError(f"Could not find a list of records. Check DATA_FIELD ('{data_field}') or response structure.")

#     if not records:
#         raise ValueError("No records found in the API response.")

#     def parse_timestamp(record: dict) -> datetime:
#         """Attempt to parse a timestamp string into a datetime object."""
#         ts = record.get(timestamp_field)
#         if ts is None:
#             raise KeyError(f"Field '{timestamp_field}' not found in record: {record}")
#         # Handles ISO 8601 format (e.g., "2026-03-06T12:00:00Z" or "2026-03-06T12:00:00+00:00")
#         return datetime.fromisoformat(ts.replace("Z", "+00:00"))

#     most_recent = max(records, key=parse_timestamp)
#     return most_recent

In [7]:
# from datetime import datetime

# get_most_recent_timestep(data, "t", 'features')

In [8]:
now_utc = datetime.now(timezone.utc)
api_t = data['features'][0]['properties']['t']
api_raw = data['features'][0]['properties']['t']
api_t = datetime.strptime(api_t, "%Y-%m-%dT%H:%M:%S")
tz = pytz.timezone('US/Eastern')
### TO DO -- run this by region and determine eastern-most timeszone programatically 
api_t = tz.localize(api_t)  ### This converts the local solar API time to Eastern. This is becuase "Eastern" is the farthest-east timezone in CONUS, and therefore subject to the longest CONUS latency if we wait for VIIRS to overpass the west coast before running. 
tz_utc = pytz.timezone('UTC') ## Now, we can convert it to UTC, because we want the UTC time of an Eastern overpass. 
api_t = api_t.astimezone(tz_utc)
time_diff = now_utc - api_t
hour_diff = (time_diff.seconds/(60))/60

some_time_buffer_thresh = 0 # Some number of hours for sensitivity.
baseline_latency = 6 # We say our latency is ~ 12 hours. We could probably get that even smaller. 
overpass_cadence = 12 ## This is the cadence of the data. We expect the data to come every 12 hours

if(hour_diff > (overpass_cadence + baseline_latency + some_time_buffer_thresh)):
    # Alert
    print(f"At {now_utc.strftime("%Y-%m-%d %H:%M:%S")} UTC, the API displayed {api_raw}. There were {round(hour_diff, 2)} hours between check time in UTC and the last API data time, or {round(hour_diff - overpass_cadence, 2)} hours since last Eastern satellite overpass. ")
  
    # make sure calling process gets an bad exit code so it bubbles as failure
    sys.exit(1)


# ## figuring this out in eastern to start
# tz = pytz.timezone('US/Eastern')
# now_utc = datetime.now(timezone.utc)
# now_est = now_utc.astimezone(tz)
# api_t = data['features'][0]['properties']['t']
# api_t = datetime.strptime(api_t, "%Y-%m-%dT%H:%M:%S")

# ### TO DO -- run this by region and determine eastern-most timeszone programatically 
# api_t = tz.localize(api_t)  ### This converts the local solar API time to Eastern. This is becuase "Eastern" is the farthest-east timezone in CONUS, and therefore subject to the longest CONUS latency if we wait for VIIRS to overpass the west coast before running. 
# #tz_utc = pytz.timezone('UTC') ## Now, we can convert it to UTC, because we want the UTC time of an Eastern overpass. 
# #api_t = api_t.astimezone(tz_utc)
# time_diff = now_utc - api_t
# hour_diff = (time_diff.seconds/(60))/60


